In [19]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Output
from scipy import stats
from matplotlib.colors import hsv_to_rgb
from IPython.display import display, clear_output

# Настройки графики
plt.style.use('ggplot')
plt.rcParams['figure.dpi'] = 100

# Инициализация сетки
size = 0.02  # ±20 мм
points = 500
x = np.linspace(-size, size, points)
y = np.linspace(-size, size, points)
X, Y = np.meshgrid(x, y)
r = np.sqrt(X**2 + Y**2)

# Создаем отдельную область для вывода
output = Output()
display(output)

def calculate_intensity(R, lambda_center_nm, delta_lambda_nm, N_spectrum=100):
    """Расчет интенсивности"""
    lambda_center = lambda_center_nm * 1e-9
    delta_lambda = delta_lambda_nm * 1e-9
    
    d = r**2 / (2 * R)
    wavelengths = np.linspace(
        max(380e-9, lambda_center - delta_lambda/2),
        min(750e-9, lambda_center + delta_lambda/2),
        N_spectrum
    )
    
    total = np.zeros_like(r)
    for wl in wavelengths:
        phase = (4 * np.pi * d) / wl
        total += 2 * (1 + np.cos(phase))
    
    return total / np.max(total)

def wavelength_to_rgb(wavelength_nm, gamma=0.5):
    """Преобразование длины волны в RGB"""
    if 380 <= wavelength_nm <= 750:
        x = (wavelength_nm - 380)/(750 - 380)
        hue = 0.7*(1 - x)
        rgb = hsv_to_rgb([hue, 1.0, 1.0])
        return rgb ** gamma
    return np.zeros(3)

@interact(
    R=FloatSlider(value=15.0, min=1.0, max=20.0, step=0.1, description="R (м)"),
    lambda_center=FloatSlider(value=550, min=400, max=700, step=1, description="λ (нм)"),
    delta_lambda=FloatSlider(value=0, min=0, max=100, step=1, description="Δλ (нм)")
)
def update_rings(R, lambda_center, delta_lambda):
    with output:
        # Очищаем вывод без прокрутки
        clear_output(wait=True)
        
        # Закрываем все предыдущие фигуры
        plt.close('all')
        
        # Расчет данных
        if delta_lambda==0:
            delta_lambda=1e-9
        intensity = calculate_intensity(R, lambda_center, delta_lambda)
        
        # Генерация RGB
        wavelengths = np.linspace(
            max(380, lambda_center - delta_lambda//2),
            min(750, lambda_center + delta_lambda//2),
            100
        )
        rgb = np.zeros((*r.shape, 3))
        for wl in wavelengths:
            color = wavelength_to_rgb(wl).reshape(1, 1, 3)
            rgb += color * intensity[..., None]
        rgb /= np.max(rgb)
        
        # Усреднение профиля
        r_flat = r.ravel()
        I_flat = intensity.ravel()
        bins = np.linspace(0, r.max(), 300)
        radial_mean, _, _ = stats.binned_statistic(r_flat, I_flat, 
                                                 statistic='mean', bins=bins)
        r_centers = (bins[:-1] + bins[1:])/2 * 1000
        
        # Создаем фигуру
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
        
        # Интерференционная картина
        ax1.imshow(rgb, 
                  extent=[-size*1000, size*1000, -size*1000, size*1000],
                  aspect='equal', 
                  origin='lower')
        ax1.set_title(f"R = {R:.1f} м\nλ = {lambda_center}±{delta_lambda} нм")
        ax1.grid(False)
        
        # Радиальный профиль
        ax2.plot(r_centers, radial_mean, 'b-', lw=2)
        ax2.grid(True)
        ax2.set_ylim(-0.1, 1.1)
        
        # Отображаем и сразу закрываем фигуру
        display(fig)
        plt.close(fig)

# Первый запуск
update_rings(15.0, 550, 0)

Output()

interactive(children=(FloatSlider(value=15.0, description='R (м)', max=20.0, min=1.0), FloatSlider(value=550.0…